[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/05%20-%20Quantum_Circuits_with_Qiskit.ipynb)


# Notebook 05 — Quantum Circuits with Qiskit

This notebook is the practical companion to Notebooks 01–04: registers, drawing, composing circuits, barriers, swapping wires, and running on the Aer simulator.


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
import numpy as np


## Registers

Name your qubits when a circuit has a meaning beyond "wire 0".


In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")
qc = QuantumCircuit(q, c)
qc.h(q[0])
qc.cx(q[0], q[1])
qc.measure(q, c)
qc.draw("mpl")


## Gates you will use constantly


In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.x(1)
qc.y(2)
qc.z(0)
qc.s(1)
qc.t(2)
qc.rx(0.4, 0)
qc.ry(0.6, 1)
qc.rz(0.8, 2)
qc.cx(0, 1)
qc.cz(1, 2)
qc.swap(0, 2)
qc.barrier()
qc.ccx(0, 1, 2)   # Toffoli
qc.draw("mpl")


## Barriers and drawing styles

`barrier` does not change the mathematics. It stops Qiskit from collapsing neighbouring gates when it draws or optimizes, which is useful for teaching.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.barrier(label="oracle")
qc.h(0)
qc.measure(0, 0)
print(qc.draw("text"))
qc.draw("mpl")


## Compose and copy

Build small pieces, then glue them. This is how oracles and adders stay readable.


In [ ]:
prep = QuantumCircuit(2, name="prep")
prep.h(0)
prep.x(1)

entangle = QuantumCircuit(2, name="bell")
entangle.cx(0, 1)

qc = QuantumCircuit(2, 2)
qc.compose(prep, inplace=True)
qc.compose(entangle, inplace=True)
qc.measure([0, 1], [0, 1])
qc.draw("mpl")


## Initialize an arbitrary state


In [ ]:
qc = QuantumCircuit(1)
qc.initialize([np.sqrt(0.2), np.sqrt(0.8)], 0)
display(qc.draw("mpl"))
print(Statevector.from_instruction(qc).probabilities_dict())


## Transpile before a backend

Real devices (and some simulators) only accept a basis gate set. `transpile` rewrites your circuit.


In [ ]:
qc = QuantumCircuit(2)
qc.swap(0, 1)
print("original depth", qc.depth())
tqc = transpile(qc, basis_gates=["cx", "u"])
display(tqc.draw("mpl"))
print("transpiled depth", tqc.depth())


## Simulator options

- default `AerSimulator()` samples measurement shots
- `AerSimulator(method="statevector")` keeps the full wavefunction when you save it


In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(qc).result()
print(result.get_statevector())


## A complete pattern: build, draw, simulate


In [ ]:
def run(circ, shots=1024):
    display(circ.draw("mpl"))
    counts = AerSimulator().run(circ, shots=shots).result().get_counts()
    display(plot_histogram(counts))
    return counts

qc = QuantumCircuit(2, 2)
qc.h([0, 1])
qc.cz(0, 1)
qc.h(1)
qc.measure([0, 1], [0, 1])
run(qc)


## Exercises

1. Write a function `bell(kind)` that returns a circuit preparing any of the four Bell states from a string `"phi+"`, `"phi-"`, `"psi+"`, `"psi-"`.
2. Compose that circuit with measurements and histogram the results.
3. Transpile a Toffoli into `cx` and `u` and count the CNOTs.


In [ ]:
# Exercise workspace
